# Limpeza dos dados de estabelecimentos

Neste procedimento faremos a limpeza dos dados dos CNPJ's obtido em: https://dados.gov.br/dados/conjuntos-dados/cadastro-nacional-da-pessoa-juridica---cnpj

In [1]:
#carga de bibliotecas

import pandas as pd # type: ignore

Foram baixados todos os dados de CNPJ's do brasil fornecidos pelo portal de dados aberto do governo federal. Com isto, temos um volume total de 10 arquivos do tipo csv com aproximadamente 1gb cada um. Apesar de um objetivo bem delimitado, faremos o armazenamento dos CNPJ's ativos em um arquivo a parte e filtraremos os dados pertinentes

In [2]:
#Local e nome dos arquivos para carregamento dos csv's da receita federal
local_entrada = "D:/bkp_humberto/PROJETOS/2024-06-24_FRETE_RETORNO/ENTRADA/CNPJs/"
local_saida = "D:/bkp_humberto/PROJETOS/2024-06-24_FRETE_RETORNO/SAIDA/"
estalecimentos = ['ESTABELE00.CSV','ESTABELE01.CSV',"ESTABELE02.CSV","ESTABELE03.CSV","ESTABELE04.CSV","ESTABELE05.CSV","ESTABELE06.CSV","ESTABELE07.CSV","ESTABELE08.CSV","ESTABELE09.CSV"]

Para compreender melhor o contexto de limpeza iniciaremos fazendo a limpeza do arquivo "ESTABELE09.CSV" e tendo compreendido o conceito os detalhes construiremos uma função para a execução dos demais arquivos

Algumas particularidades do dado bruto: Não possui títulos das colunas, separador utilizado é o ";", o tipo de codificação do arquivo é o europeu ocidental 1250 ou cp1250

In [3]:
#carga do arquivo por meio do pandas
df_estabele09_sujo = pd.read_csv(local_entrada +estalecimentos[9],sep=';' ,encoding='cp1250',header=None)

C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\124217507.py:2: DtypeWarning: Columns (8,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df_estabele09_sujo = pd.read_csv(local_entrada +estalecimentos[9],sep=';' ,encoding='cp1250',header=None)


Faremos a listagem dos nomes das colunas do dataframe

In [4]:
#colunas do dataframe
colunasdf = df_estabele09_sujo.columns
print(colunasdf)

Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
      dtype='int64')


Antes de iniciar o processo de limpeza precisaremos renomear as colunas para facilitar o trabalho com o dado em quesão, para isto utilizaremos o metadado disponibilidazo na fonte do dado e montaremos um mapeamento de chave-valor para troca correta das informações

In [5]:
#conjunto cahve-valor para troca do dado
map_titulos = {
    0: 'CNPJ BÁSICO',
    1:'CNPJ ORIGEM',
    2:'CNJP DV',
    3:'IDENTIFICADOR MATRIZ FILIAL',
    4:'NOME FANTASIA',
    5:'SITUACAO CADASTRAL',
    6:'DATA SITUACAO CADASTRAL',
    7:'MOTIVO SITUACAO CADASTRAL',
    8:'NOME DA CIDADE NO EXTERIOR',
    9:'PAIS',
    10:'DATA DE INICIO ATIVIDADE',
    11:'CNAE FISCAL PRINCIPAL',
    12:'CNAE FISCAL SECUNDARIO',
    13:'TIPO DE LOGRADOURO',
    14:'LOGRADOURO',
    15:'NUMERO',
    16:'COMPLEMENTO',
    17:'BAIRRO',
    18:'CEP',
    19:'UF',
    20:'MUNICIPIO',
    21:'DDD1',
    22:'TELEFONE 1',
    23:'DDD2',
    24:'TELEFONE 2',
    25:'DDD DO FAX',
    26:'FAX',
    27:'CORREIO ELETRONICO',
    28:'SITUACAO ESPECIAL',
    29:'DATA DA SITUACAO ESPECIAL'
}

In [6]:
#troca dos valores numéricos das colunas pelos nomes atribuidos no metadado
df_estabele09_sujo.rename(columns=map_titulos, inplace=True)

#verificando troca dos nomes das colunas
colunasdf = df_estabele09_sujo.columns
print(colunasdf)

Index(['CNPJ BÁSICO', 'CNPJ ORIGEM', 'CNJP DV', 'IDENTIFICADOR MATRIZ FILIAL',
       'NOME FANTASIA', 'SITUACAO CADASTRAL', 'DATA SITUACAO CADASTRAL',
       'MOTIVO SITUACAO CADASTRAL', 'NOME DA CIDADE NO EXTERIOR', 'PAIS',
       'DATA DE INICIO ATIVIDADE', 'CNAE FISCAL PRINCIPAL',
       'CNAE FISCAL SECUNDARIO', 'TIPO DE LOGRADOURO', 'LOGRADOURO', 'NUMERO',
       'COMPLEMENTO', 'BAIRRO', 'CEP', 'UF', 'MUNICIPIO', 'DDD1', 'TELEFONE 1',
       'DDD2', 'TELEFONE 2', 'DDD DO FAX', 'FAX', 'CORREIO ELETRONICO',
       'SITUACAO ESPECIAL', 'DATA DA SITUACAO ESPECIAL'],
      dtype='object')


Após a leitura do metadados disponibilizados defini quais colunas não são úteis na minha análise, são elas:

1. DATA SITUACAO CADASTRAL
2. MOTIVO SITUACAO CADASTRAL
3. NOME DA CIDADE NO EXTERIOR
4. DATA DE INICIO ATIVIDADE
5. DDD DO FAX
6. FAX
7. SITUACAO ESPECIAL
8. DATA DA SITUACAO ESPECIAL


Em seguida faremos o drop de todas estas colunas.

In [7]:
#drop das colunas
df_estabele09_dropado = df_estabele09_sujo.drop(['DATA SITUACAO CADASTRAL','MOTIVO SITUACAO CADASTRAL','NOME DA CIDADE NO EXTERIOR','DATA DE INICIO ATIVIDADE','DDD DO FAX','FAX','SITUACAO ESPECIAL','DATA DA SITUACAO ESPECIAL'],axis=1)

Feito o drop das colunas não interesasntes, filtraremos a coluna que trata da situação cadastral das empresas. Nossa análise busca as empresas com situação cadastral ativa, neste caso as empresas ativas possuem o código 2 na coluna de índice 'SITUACAO CADASTRAL' do nosso dataframe.

In [8]:
#filtro de situação cadastral da empresa atraves de uma mascara
mascara_ativos = df_estabele09_dropado['SITUACAO CADASTRAL'] == 2

df_estabele09_ativos = df_estabele09_dropado[mascara_ativos]

Realizado o filtro das empresas ativas, faremos o filtro das empresas pertencentes ao estado de Pernambuco fintrando pela coluna 'UF'

In [9]:
#filtro para empreendimentos situados em pernambuco
mascara_UF = df_estabele09_ativos['UF'] == 'PE'

df_estabele09_ATV_PE = df_estabele09_ativos[mascara_UF]

Por fim salvaremos o dataframe  limpo para análise posterior


In [10]:
df_estabele09_ATV_PE.to_csv(local_saida + estalecimentos[9],sep=';' ,index=False )

Tendo em vista que todo o procendimento de limpeza dos dados resultou em uma infoirmação consistente. Construiremos uma função para realizar o mesmo processo de limpeza em todos os demais arquivos do tipo csv referentes aos estabelecimentos re salvaremos o dados

In [22]:
#Definindo a função e dados de entrada
def limpeza_estabelecimentos(entrada, saida, dataframe, mapa):
    
    #carga do arquivo
    df = pd.read_csv(entrada +dataframe ,sep=';' ,encoding='cp1250',header=None)

    #troca dos valores numéricos das colunas pelos nomes atribuidos no metadado
    df.rename(columns=mapa, inplace=True)

    #drop das colunas
    df_drop = df.drop(['DATA SITUACAO CADASTRAL','MOTIVO SITUACAO CADASTRAL','NOME DA CIDADE NO EXTERIOR','DATA DE INICIO ATIVIDADE','DDD DO FAX','FAX','SITUACAO ESPECIAL','DATA DA SITUACAO ESPECIAL'],axis=1)

    #filtro de situacao
    filtro_situacao = df['SITUACAO CADASTRAL'] == 2
    df_situacao = df[filtro_situacao]

    #filtro de estado
    filtro_UF = df_situacao['UF'] =='PE'
    df_final = df_situacao[filtro_UF]

    #save do arquivo final
    df_final.to_csv(saida+dataframe,sep=';', index=False)



Tendo em vista que o arquivo 00 é muito maior que os demais ele será tratado a parte e os demais estarão serão tratadas juntos em um FOR

In [24]:
#Faremos um for para executar a função em todos os arquivos

for i in range(8):
    limpeza_estabelecimentos(local_entrada,local_saida,estalecimentos[(i+1)],map_titulos)

C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\327568765.py:5: DtypeWarning: Columns (8,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entrada +dataframe ,sep=';' ,encoding='cp1250',header=None)
C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\327568765.py:5: DtypeWarning: Columns (8,21,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entrada +dataframe ,sep=';' ,encoding='cp1250',header=None)
C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\327568765.py:5: DtypeWarning: Columns (8,21,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entrada +dataframe ,sep=';' ,encoding='cp1250',header=None)
C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\327568765.py:5: DtypeWarning: Columns (8,18,21,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entrada +dataframe ,

In [25]:
limpeza_estabelecimentos(local_entrada,local_saida,estalecimentos[0],map_titulos)

C:\Users\hcage\AppData\Local\Temp\ipykernel_15364\327568765.py:5: DtypeWarning: Columns (8,21,22,24,26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(entrada +dataframe ,sep=';' ,encoding='cp1250',header=None)
